# Evaluate a saved FLUX checkpoint on a fixed ARC training view

This inference-only Modal experiment measures whether a standalone trained
FLUX transformer can reconstruct targets from augmentation prefixes that were
actually presented during training. The default panel selects the first
`train` key for each task, then evaluates the final target prefix of that key.

The model-facing prompt is reconstructed without its target output and is
required to match the exact token prefix used during training. Expected grids
remain separate until scoring.

Prerequisites:

- a CUDA GPU with at least 70 GiB VRAM;
- the writable `diffusion-arc-models` Volume mounted at
  `/mnt/diffusion-arc-models`;
- a completed training run whose updated `report.json` records the saved
  transformer and its hashes;
- a Modal Secret containing `GITHUB_TOKEN`;
- `DIFFUSION_ARC_REVISION` set to the exact pushed evaluation-code commit
  (default: `main`).

Future checkpoint evaluations normally require changes only in the first
configuration cell. `DIFFUSION_ARC_VIEW_NAME` may be `train`, `train_unseen`,
or `train_original`.


In [ ]:
!nvidia-smi


In [ ]:
%uv pip uninstall torchvision


## 1. Immutable evaluation configuration

The defaults point to the first overfitting pilot. Override the corresponding
environment variables or edit these values before executing this cell.


In [ ]:
from __future__ import annotations

import base64
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import sys
import textwrap
import time

TRAINING_RUN_ID = os.environ.get(
    'DIFFUSION_ARC_TRAINING_RUN_ID',
    '20260825T060523831464Z',
).strip()
CHECKPOINT_NAME = os.environ.get(
    'DIFFUSION_ARC_CHECKPOINT_NAME',
    'flux-transformer-step-0832',
).strip()
VIEW_NAME = os.environ.get('DIFFUSION_ARC_VIEW_NAME', 'train').strip()
SOURCE_REVISION = os.environ.get('DIFFUSION_ARC_REVISION', 'main').strip()
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'

if not TRAINING_RUN_ID or not CHECKPOINT_NAME or not SOURCE_REVISION:
    raise ValueError('Run ID, checkpoint name, and source revision must be non-empty')
if VIEW_NAME not in {'train', 'train_unseen', 'train_original'}:
    raise ValueError(
        "DIFFUSION_ARC_VIEW_NAME must be 'train', 'train_unseen', or 'train_original'"
    )

GENERATION_SEEDS = (167, 168)
INFERENCE_STEPS = 50
EXPECTED_TASKS = 8
MINIMUM_VRAM_GIB = 70.0
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
BASE_FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
MODEL_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
ARC_DATA_ROOT = VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2'
ARC_DATA_MANIFEST = VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json'
TRAINING_RUN_ROOT = (
    VOLUME_ROOT
    / 'results'
    / 'single-gpu-overfitting-pilot'
    / TRAINING_RUN_ID
)
TRAINING_REPORT_PATH = TRAINING_RUN_ROOT / 'report.json'
TRANSFORMER_CHECKPOINT = TRAINING_RUN_ROOT / CHECKPOINT_NAME

EVALUATION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EVALUATION_ROOT = (
    TRAINING_RUN_ROOT
    / 'checkpoint-evaluations'
    / VIEW_NAME
    / EVALUATION_ID
)
EVALUATION_REPORT_PATH = EVALUATION_ROOT / 'report.json'
IMAGES_ROOT = EVALUATION_ROOT / 'generated_images'

required_paths = (
    BASE_FLUX_CHECKPOINT,
    NVARC_CHECKPOINT,
    MODEL_MANIFEST,
    ARC_DATA_ROOT,
    ARC_DATA_MANIFEST,
    TRAINING_REPORT_PATH,
    TRANSFORMER_CHECKPOINT,
)
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing_paths}')

EVALUATION_ROOT.mkdir(parents=True, exist_ok=False)
IMAGES_ROOT.mkdir()

EVALUATION: dict[str, object] = {
    'schema_version': 1,
    'status': 'INITIALIZING',
    'evaluation_id': EVALUATION_ID,
    'question': (
        'Can the saved FLUX checkpoint exactly reconstruct final targets from '
        'augmentation prefixes selected from a configured training view?'
    ),
    'configuration': {
        'training_run_id': TRAINING_RUN_ID,
        'checkpoint_name': CHECKPOINT_NAME,
        'view_name': VIEW_NAME,
        'generation_seeds': list(GENERATION_SEEDS),
        'inference_steps': INFERENCE_STEPS,
        'requested_source_revision': SOURCE_REVISION,
        'automatic_shutdown': AUTO_SHUTDOWN,
    },
    'paths': {
        'training_report': str(TRAINING_REPORT_PATH),
        'transformer_checkpoint': str(TRANSFORMER_CHECKPOINT),
        'evaluation_report': str(EVALUATION_REPORT_PATH),
        'images': str(IMAGES_ROOT),
    },
    'failures': [],
}


def save_evaluation() -> None:
    temporary = EVALUATION_REPORT_PATH.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(EVALUATION, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(EVALUATION_REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    EVALUATION['status'] = 'FAIL'
    EVALUATION['failed_stage'] = stage
    failures = EVALUATION.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append({
            'stage': stage,
            'type': type(error).__name__,
            'message': str(error),
            'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
        })
    save_evaluation()


def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


save_evaluation()
print('Training run:', TRAINING_RUN_ID)
print('Transformer:', TRANSFORMER_CHECKPOINT)
print('Evaluation view:', VIEW_NAME)
print('Evaluation report:', EVALUATION_REPORT_PATH)


## 2. Fetch and verify the exact evaluation source

The notebook records both the original training source revision and the newer
evaluation source revision. The latter may add inference utilities without
changing the trained checkpoint.


In [ ]:
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-checkpoint-eval-{EVALUATION_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    error = RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    record_failure('source-authentication', error)
    raise error
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del git_environment
    del credential

RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--no-input',
        '--no-deps',
        '--no-build-isolation',
        '-e',
        str(SOURCE_DIR),
    ],
)

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
test_seconds = time.perf_counter() - test_started
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    error = RuntimeError(f'Unit tests returned {test_process.returncode}')
    record_failure('unit-tests', error)
    raise error

EVALUATION['evaluation_source'] = {
    'repository': REPOSITORY_URL,
    'requested_revision': SOURCE_REVISION,
    'resolved_commit': RESOLVED_SOURCE_COMMIT,
    'path': str(SOURCE_DIR),
}
EVALUATION['packages'] = {name: installed_version(name) for name in PINNED_PACKAGES}
EVALUATION['unit_tests'] = {'returncode': 0, 'seconds': test_seconds}
save_evaluation()
print('Evaluation source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {test_seconds:.2f}s')


## 3. Validate GPU, source data, split, report, and checkpoint hashes

The trained transformer is standalone. NVARC and the VAE still come from the
original verified model Volume paths. The selected split files must match the
hashes recorded by the training run.


In [ ]:
import torch

try:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')
    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    properties = torch.cuda.get_device_properties(DEVICE)
    gpu_vram_gib = properties.total_memory / 2**30
    if gpu_vram_gib < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires at least {MINIMUM_VRAM_GIB:.0f} GiB VRAM; '
            f'received {properties.name} with {gpu_vram_gib:.1f} GiB'
        )
except Exception as error:
    record_failure('cuda-contract', error)
    raise


def cuda_memory(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
    }


training_report = json.loads(TRAINING_REPORT_PATH.read_text(encoding='utf-8'))
if training_report.get('status') != 'COMPLETE':
    raise RuntimeError('Training report is not COMPLETE')
saved_checkpoint = training_report.get('saved_checkpoint')
if not isinstance(saved_checkpoint, dict):
    raise RuntimeError('Training report does not record a saved_checkpoint')
if Path(saved_checkpoint.get('path', '')).resolve() != TRANSFORMER_CHECKPOINT.resolve():
    raise RuntimeError('Selected transformer path differs from training report')

expected_checkpoint_files = saved_checkpoint.get('files')
if not isinstance(expected_checkpoint_files, dict) or not expected_checkpoint_files:
    raise RuntimeError('Training report contains no checkpoint file manifest')
actual_checkpoint_files = {
    str(path.relative_to(TRANSFORMER_CHECKPOINT)): path
    for path in sorted(TRANSFORMER_CHECKPOINT.rglob('*'))
    if path.is_file()
}
if set(actual_checkpoint_files) != set(expected_checkpoint_files):
    raise RuntimeError('Checkpoint file set differs from training report')
for relative, path in actual_checkpoint_files.items():
    expected = expected_checkpoint_files[relative]
    if path.stat().st_size != expected['bytes'] or sha256_file(path) != expected['sha256']:
        raise RuntimeError(f'Checkpoint file differs: {relative}')

if sha256_file(MODEL_MANIFEST) != training_report['model_manifest_sha256']:
    raise RuntimeError('Model manifest differs from training run')
if sha256_file(ARC_DATA_MANIFEST) != training_report['dataset']['manifest_sha256']:
    raise RuntimeError('Dataset manifest differs from training run')
dataset_manifest = json.loads(ARC_DATA_MANIFEST.read_text(encoding='utf-8'))
for task_id in training_report['dataset']['task_ids']:
    relative = f'training/{task_id}.json'
    task_path = ARC_DATA_ROOT / relative
    expected_hash = dataset_manifest['files'].get(relative)
    if expected_hash is None or not task_path.is_file():
        raise RuntimeError(f'Dataset manifest lacks selected task: {relative}')
    if sha256_file(task_path) != expected_hash:
        raise RuntimeError(f'Selected raw task differs: {relative}')

TEMP_ROOT = Path('/tmp') / f'diffusion-arc-checkpoint-eval-data-{EVALUATION_ID}'
TEMP_DATA_ROOT = TEMP_ROOT / 'data'
RUNTIME_SPLIT_ROOT = TEMP_DATA_ROOT / 'splits' / 'overfitting_run_1'
RUNTIME_SPLIT_ROOT.parent.mkdir(parents=True)
shutil.copytree(
    SOURCE_DIR / 'data' / 'splits' / 'overfitting_run_1',
    RUNTIME_SPLIT_ROOT,
)
(TEMP_DATA_ROOT / 'raw').mkdir()
os.symlink(ARC_DATA_ROOT, TEMP_DATA_ROOT / 'raw' / 'arc-agi-2', target_is_directory=True)

actual_split_hashes = {
    str(path.relative_to(RUNTIME_SPLIT_ROOT)): sha256_file(path)
    for path in sorted(RUNTIME_SPLIT_ROOT.rglob('*.json'))
}
if actual_split_hashes != training_report['dataset']['split_file_sha256']:
    raise RuntimeError('Evaluation split files differ from training run')

from diffusion_arc.training.config import load_single_gpu_training_config

source_config = load_single_gpu_training_config(
    SOURCE_DIR / 'configs' / 'training' / 'single_gpu_overfitting_pilot.json'
)
RUN_CONFIG = replace(
    source_config,
    models=replace(
        source_config.models,
        flux_checkpoint=BASE_FLUX_CHECKPOINT,
        nvarc_checkpoint=NVARC_CHECKPOINT,
    ),
    data=replace(source_config.data, split_directory=RUNTIME_SPLIT_ROOT),
)

EVALUATION['training_source'] = training_report['source']
EVALUATION['training_checkpoint'] = {
    'global_step': saved_checkpoint['global_step'],
    'path': str(TRANSFORMER_CHECKPOINT),
    'files': expected_checkpoint_files,
}
EVALUATION['environment'] = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu_name': properties.name,
    'gpu_vram_gib': gpu_vram_gib,
    'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
    'bf16_supported': torch.cuda.is_bf16_supported(),
}
EVALUATION['split_file_sha256'] = actual_split_hashes
EVALUATION['resolved_model_config'] = json.loads(
    json.dumps(asdict(RUN_CONFIG.models), default=str)
)
save_evaluation()
print(f'GPU: {properties.name} | {gpu_vram_gib:.1f} GiB')
print('Checkpoint files verified:', len(actual_checkpoint_files))


## 4. Load frozen NVARC/VAE and the saved standalone transformer

No optimizer is created. All models are frozen after their checkpoint
contracts have been validated.


In [ ]:
from diffusion_arc.training.single_gpu_setup import (
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = prepare_single_gpu_device(RUN_CONFIG)
seed_single_gpu_process(GENERATION_SEEDS[0])
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

load_started = time.perf_counter()
models = load_single_gpu_models(
    RUN_CONFIG,
    device=device,
    transformer_checkpoint=TRANSFORMER_CHECKPOINT,
)
models.transformer.requires_grad_(False)
load_seconds = time.perf_counter() - load_started

if any(parameter.requires_grad for parameter in models.nvarc_runner.parameters()):
    raise RuntimeError('NVARC unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.vae_decoder.parameters()):
    raise RuntimeError('VAE unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.transformer.parameters()):
    raise RuntimeError('Evaluation transformer unexpectedly trainable')

EVALUATION['models'] = {
    'load_seconds': load_seconds,
    'transformer_parameters': sum(
        parameter.numel() for parameter in models.transformer.parameters()
    ),
    'memory_after_load': cuda_memory('after-model-load'),
}
save_evaluation()
EVALUATION['models']


## 5. Reconstruct one exact training prefix per task

For each task, the first key in the selected view is materialized. The final
pair becomes the query and every earlier pair remains a demonstration. A token
equality assertion proves that the target-free evaluation prompt is exactly
the prefix used for that training target.


In [ ]:
from diffusion_arc.arc_data import (
    ArcEvaluationTask,
    ArcQuery,
    apply_training_augmentation,
    load_training_task,
    render_evaluation_prompts,
    tokenize_nvarc_evaluation_sample,
)
from diffusion_arc.inference import prepare_evaluation_text_inputs
from diffusion_arc.training import ArcTrainingDataset

dataset = ArcTrainingDataset(
    split_directory=RUNTIME_SPLIT_ROOT,
    view_name=VIEW_NAME,
    tokenizer=models.tokenizer,
)
expected_task_ids = tuple(training_report['dataset']['task_ids'])
selected_keys = []
evaluation_samples = []
expected_grids = []
seen_task_ids = set()

for index, key in enumerate(dataset.keys):
    if key.task_id in seen_task_ids:
        continue
    training_sample = dataset[index]
    raw_task = load_training_task(dataset.source_directory / f'{key.task_id}.json')
    augmented = apply_training_augmentation(raw_task, key.augmentation)
    if len(augmented.pairs) < 2:
        raise RuntimeError(f'Selected key has fewer than two pairs: {key}')

    query_pair = augmented.pairs[-1]
    evaluation_task = ArcEvaluationTask(
        task_id=str(key),
        demonstrations=augmented.pairs[:-1],
        queries=(ArcQuery(query_index=0, input_grid=query_pair.input_grid),),
    )
    rendered = render_evaluation_prompts(evaluation_task)[0]
    tokenized = tokenize_nvarc_evaluation_sample(
        rendered,
        models.tokenizer,
        max_tokens=dataset.max_tokens,
    )
    training_prefix = training_sample.input_ids[:training_sample.prefix_token_ends[-1]]
    if tokenized.input_ids != training_prefix:
        raise RuntimeError(f'Evaluation tokens differ from training prefix: {key}')

    selected_keys.append(str(key))
    evaluation_samples.append(tokenized)
    expected_grids.append(query_pair.output_grid)
    seen_task_ids.add(key.task_id)

if tuple(key.split('_', 1)[0] for key in selected_keys) != expected_task_ids:
    raise RuntimeError(
        f'Selected task order differs: {selected_keys!r} versus {expected_task_ids!r}'
    )
if len(selected_keys) != EXPECTED_TASKS:
    raise RuntimeError(
        f'Expected {EXPECTED_TASKS} selected keys, received {len(selected_keys)}'
    )

text_inputs = prepare_evaluation_text_inputs(
    evaluation_samples,
    nvarc_runner=models.nvarc_runner,
)
EVALUATION['panel'] = {
    'view_name': VIEW_NAME,
    'keys': selected_keys,
    'target_free_token_counts': [len(sample.input_ids) for sample in evaluation_samples],
    'expected_shapes': [[len(grid), len(grid[0])] for grid in expected_grids],
    'exact_training_prefix_match': [True] * len(selected_keys),
    'memory_after_conditioning': cuda_memory('after-conditioning'),
}
save_evaluation()
EVALUATION['panel']


## 6. Generate, score, and compare with the canonical checkpoint boundary

The selected panel is generated with the same two seeds and 50-step linear
Euler schedule used in the training notebook. Images and decoded grids are
stored in the evaluation directory.


In [ ]:
from PIL import Image
from tqdm.auto import tqdm

from diffusion_arc.inference import (
    diagnose_exact_grids,
    generate_arc_batch,
    linear_timestep_schedule,
)


def diagnostic_payload(diagnostic) -> dict[str, object]:
    return {
        'prediction_valid': diagnostic.prediction_valid,
        'invalid_reason': diagnostic.invalid_reason,
        'predicted_shape': diagnostic.predicted_shape,
        'expected_shape': diagnostic.expected_shape,
        'shape_match': diagnostic.shape_match,
        'cell_accuracy': diagnostic.cell_accuracy,
        'exact_grid': diagnostic.exact_grid,
    }


schedule = linear_timestep_schedule(INFERENCE_STEPS, device=device)
samples_by_seed = []
diagnostics_by_seed = []
generation_started = time.perf_counter()
try:
    for generation_seed in tqdm(
        GENERATION_SEEDS,
        desc=f'Checkpoint inference ({VIEW_NAME})',
        unit='seed',
    ):
        result = generate_arc_batch(
            models.transformer,
            models.vae_decoder,
            text_inputs,
            timestep_schedule=schedule,
            compute_dtype=models.flux_compute_dtype,
            generator=torch.Generator(device=device).manual_seed(generation_seed),
        )
        diagnostics = diagnose_exact_grids(result.predictions, expected_grids)
        seed_samples = []
        for index, (key, prediction, diagnostic) in enumerate(
            zip(selected_keys, result.predictions, diagnostics, strict=True)
        ):
            task_id = key.split('_', 1)[0]
            filename = f'{index:02d}-{task_id}-seed-{generation_seed}.png'
            image = result.rgb_images[index].permute(1, 2, 0).contiguous().numpy()
            Image.fromarray(image).save(IMAGES_ROOT / filename)
            seed_samples.append({
                'key': key,
                'seed': generation_seed,
                'image': str((IMAGES_ROOT / filename).relative_to(EVALUATION_ROOT)),
                'prediction': prediction.grid,
                'diagnostic': diagnostic_payload(diagnostic),
            })
        samples_by_seed.append(seed_samples)
        diagnostics_by_seed.append(diagnostics)
except Exception as error:
    record_failure('generation', error)
    raise

exact_pass1 = sum(item.exact_grid for item in diagnostics_by_seed[0])
exact_pass2 = sum(
    first.exact_grid or second.exact_grid
    for first, second in zip(*diagnostics_by_seed, strict=True)
)
valid_pass2 = sum(
    first.prediction_valid or second.prediction_valid
    for first, second in zip(*diagnostics_by_seed, strict=True)
)
shape_pass2 = sum(
    first.shape_match or second.shape_match
    for first, second in zip(*diagnostics_by_seed, strict=True)
)
best_cell_accuracies = []
for first, second in zip(*diagnostics_by_seed, strict=True):
    available = [
        value for value in (first.cell_accuracy, second.cell_accuracy)
        if value is not None
    ]
    best_cell_accuracies.append(max(available) if available else None)

seen_metrics = {
    'exact_grid_pass1': exact_pass1,
    'exact_grid_pass2': exact_pass2,
    'valid_grid_pass2': valid_pass2,
    'correct_shape_pass2': shape_pass2,
    'best_cell_accuracies': best_cell_accuracies,
}
checkpoint_step = saved_checkpoint['global_step']
canonical_entry = next(
    (
        entry
        for entry in training_report['generation']
        if entry['global_step'] == checkpoint_step
    ),
    None,
)
if canonical_entry is None:
    raise RuntimeError(
        f'Training report has no canonical generation at step {checkpoint_step}'
    )
canonical_metrics = {
    key: canonical_entry[key]
    for key in (
        'exact_grid_pass1',
        'exact_grid_pass2',
        'valid_grid_pass2',
        'correct_shape_pass2',
        'best_cell_accuracies',
    )
}

EVALUATION['generation'] = {
    'seconds': time.perf_counter() - generation_started,
    'samples_by_seed': samples_by_seed,
    'metrics': seen_metrics,
    'memory': cuda_memory('generation-complete'),
}
EVALUATION['comparison'] = {
    'selected_view': seen_metrics,
    'canonical_original': canonical_metrics,
    'delta': {
        'exact_grid_pass2': (
            seen_metrics['exact_grid_pass2'] - canonical_metrics['exact_grid_pass2']
        ),
        'valid_grid_pass2': (
            seen_metrics['valid_grid_pass2'] - canonical_metrics['valid_grid_pass2']
        ),
        'correct_shape_pass2': (
            seen_metrics['correct_shape_pass2'] - canonical_metrics['correct_shape_pass2']
        ),
    },
}
EVALUATION['status'] = 'COMPLETE'
save_evaluation()
print(json.dumps(EVALUATION['comparison'], indent=2, allow_nan=False))
print('Evaluation report:', EVALUATION_REPORT_PATH)


## 7. Interpretation boundary

- Better results on `train` than on the canonical panel support memorization of
  seen transformed prefixes.
- Similar `train` and canonical results indicate that the checkpoint has not
  preferentially fit those exact training prefixes.
- A later `train_unseen` run separates memorization from augmentation-level
  generalization.

The final cell optionally stops the current Modal container ten seconds after
the completed report has been flushed to the Volume.


In [ ]:
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; stop the kernel manually when finished.')
else:
    if EVALUATION.get('status') != 'COMPLETE':
        raise RuntimeError(
            f"Refusing shutdown because evaluation status is {EVALUATION.get('status')!r}"
        )
    container_id = os.environ.get('MODAL_TASK_ID', '')
    modal_cli = shutil.which('modal')
    if not re.fullmatch(r'ta-[A-Za-z0-9-]+', container_id):
        raise RuntimeError(f'Invalid or missing MODAL_TASK_ID: {container_id!r}')
    if modal_cli is None:
        raise RuntimeError('Modal CLI was not found')

    shutdown_log = EVALUATION_ROOT / 'shutdown-request.log'
    EVALUATION['automatic_shutdown'] = {
        'container_id': container_id,
        'delay_seconds': 10,
        'log': str(shutdown_log),
    }
    save_evaluation()
    os.sync()

    shutdown_script = textwrap.dedent(
        f'''
        import subprocess
        import time

        time.sleep(10)
        with open({str(shutdown_log)!r}, 'a', encoding='utf-8') as log:
            log.write('Requesting Modal container shutdown.\\n')
            log.flush()
            subprocess.run(
                [{modal_cli!r}, 'container', 'stop', '--yes', {container_id!r}],
                stdout=log,
                stderr=subprocess.STDOUT,
                text=True,
                check=False,
            )
        '''
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_script],
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
        close_fds=True,
    )
    print('Evaluation vollständig gespeichert.')
    print(f'Modal-Container {container_id} wird in 10 Sekunden beendet.')
